In [ ]:
import pandas as pd
from sqlalchemy import text
from connection import connect
from translate_language import convert_language

# Conexion
co_oltp, etl_conn, etl_conn_or = connect()

# Extraccion de datos desde el OLTP
# Se basa en humanresources.employee + person.person + department

query_employee = text("""
SELECT
    e.business_entity_id AS employee_alternate_key,
    p.title,
    p.first_name,
    p.middle_name,
    p.last_name,
    p.suffix,
    e.gender,
    e.marital_status,
    e.birth_date,
    e.hire_date,
    e.salaried_flag,
    e.vacation_hours,
    e.sick_leave_hours,
    e.current_flag,
    e.organization_level,
    e.job_title,
    e.login_id,
    ea.email_address,
    pp.phone_number AS phone,
    d.name AS department_name,
    h.start_date,
    h.end_date
FROM humanresources.employee AS e
INNER JOIN person.person AS p
    ON e.business_entity_id = p.business_entity_id
LEFT JOIN person.email_address AS ea
    ON p.business_entity_id = ea.business_entity_id
LEFT JOIN person.person_phone AS pp
    ON p.business_entity_id = pp.business_entity_id
LEFT JOIN humanresources.employee_department_history AS h
    ON e.business_entity_id = h.business_entity_id
LEFT JOIN humanresources.department AS d
    ON h.department_id = d.department_id
""")

df_emp = pd.read_sql(query_employee, co_oltp)
print(f"Registros extraidos: {len(df_emp)}")
print(df_emp.head(3))

# Enlaza con DimSalesTerritory (territorio de ventas)
df_terr = pd.read_sql(
    text("SELECT sales_territory_key, sales_territory_alternate_key FROM dim_sales_territory;"),
    etl_conn
)

# Algunos empleados pueden tener territorio asignado si son vendedores
# (se usa la tabla sales.sales_person)
sales_person = pd.read_sql(
    text("SELECT business_entity_id AS employee_alternate_key, territory_id FROM sales.sales_person;"),
    co_oltp
)

df_emp = df_emp.merge(
    sales_person,
    on='employee_alternate_key',
    how='left'
)

df_emp = df_emp.merge(
    df_terr,
    left_on='territory_id',
    right_on='sales_territory_alternate_key',
    how='left'
).drop(['territory_id', 'sales_territory_alternate_key'], axis=1)

# Transformaciones
df_emp['name_style'] = 0
df_emp['sales_person_flag'] = df_emp['sales_territory_key'].notnull().astype(int)
df_emp['current_flag'] = df_emp['current_flag'].astype(int)

# Limpia columnas textuales nulas
for col in ['department_name', 'title', 'job_title']:
    df_emp[col] = df_emp[col].fillna('Unknown')

# Traduccion opcional de departamentos
df_emp = convert_language('en', 'es', 'department_name', 'department_name_es', df_emp)
df_emp = convert_language('en', 'fr', 'department_name', 'department_name_fr', df_emp)

# Selecciona las columnas finales 
final_columns = [
    'employee_alternate_key',
    'sales_territory_key',
    'first_name',
    'last_name',
    'middle_name',
    'title',
    'gender',
    'marital_status',
    'birth_date',
    'hire_date',
    'login_id',
    'email_address',
    'phone',
    'salaried_flag',
    'vacation_hours',
    'sick_leave_hours',
    'current_flag',
    'sales_person_flag',
    'department_name',
    'start_date',
    'end_date'
]

df_to_load = df_emp[final_columns]
print("Columnas finales:", df_to_load.columns.tolist())
print(df_to_load.head(5))

# Carga al DW
df_to_load.to_sql(
    'dim_employee',
    etl_conn,
    if_exists='append',
    index=False
)

print("Carga finalizada en DimEmployee")
